# QuantGAN - Component Tests

This notebook tests all major components of the QuantGAN package to ensure everything works correctly.

**What we test:**
1. ✅ Imports & Package Installation
2. ✅ Configuration Classes
3. ✅ Data Loading & Preprocessing
4. ✅ Dataset Building
5. ✅ Model Building (both generators + discriminator)
6. ✅ Training Components
7. ✅ Evaluation & Metrics
8. ✅ Utilities (I/O, Inference, Random)
9. ✅ Quick Mini-Training (5 epochs)

**Expected runtime:** ~3-5 minutes

## Setup

In [1]:
import os
import sys
import numpy as np
import tensorflow as tf
from scipy.stats import kurtosis

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

2026-02-14 00:11:04.424436: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-14 00:11:04.425111: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-14 00:11:04.534567: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-14 00:11:06.915086: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

Python: 3.10.19 | packaged by conda-forge | (main, Jan 26 2026, 23:45:08) [GCC 14.3.0]
NumPy: 2.2.6
TensorFlow: 2.20.0
GPUs: []


2026-02-14 00:11:07.951270: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Test 1: Package Import ✅

Verify that the package is installed and all modules can be imported.

In [7]:
try:
    # Core imports
    import quantgan
    from quantgan import (
        DataConfig,
        PreprocessConfig,
        DatasetConfig,
        ModelConfig,
        TrainConfig,
        DebugConfig,
        RunConfig,
    )
    
    # Data imports
    from quantgan.data import (
        DefeatBetaSource,
        LambertWPreprocessor,
        DatasetBuilder,
    )
    
    # Model imports
    from quantgan.models import (
        build_generator,
        build_discriminator,
        TCNBlock,
    )
    
    # Training imports
    from quantgan.training import WGANGPTrainer, EpochDecay
    
    # Evaluation imports
    from quantgan.evaluation import (
        PaperEvaluator,
        Plotter,
        acf_tf,
        leverage_tf,
        acf_vec,
        lev_vec,
    )
    
    # Utils imports
    from quantgan.utils import (
        set_all_seeds,
        build_and_load_generator,
        generate_M_paths_raw,
        write_weights_meta,
    )
    
    print("✅ All imports successful!")
    print(f"   Package version: quantgan {quantgan.__version__ if hasattr(quantgan, '__version__') else '0.1.0'}")
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("   Make sure you ran: pip install -e .")
    raise

✅ All imports successful!
   Package version: quantgan 0.1.0


## Test 2: Configuration Classes ✅

Test that all config classes work correctly.

In [8]:
# Test creating configs with defaults
data_cfg = DataConfig()
pre_cfg = PreprocessConfig()
ds_cfg = DatasetConfig()
model_cfg = ModelConfig()
train_cfg = TrainConfig()
debug_cfg = DebugConfig()
run_cfg = RunConfig()

print("✅ Default configs created")
print(f"   DataConfig.ticker: {data_cfg.ticker}")
print(f"   ModelConfig.generator_type: {model_cfg.generator_type}")
print(f"   TrainConfig.epochs: {train_cfg.epochs}")

# Test creating configs with custom values
custom_data = DataConfig(ticker="AAPL", start="2020-01-01")
custom_model = ModelConfig(generator_type="svnn", z_dim=5)

print("\n✅ Custom configs created")
print(f"   Custom ticker: {custom_data.ticker}")
print(f"   Custom generator: {custom_model.generator_type}")
print(f"   Custom z_dim: {custom_model.z_dim}")

✅ Default configs created
   DataConfig.ticker: SPY
   ModelConfig.generator_type: pure_tcn
   TrainConfig.epochs: 200

✅ Custom configs created
   Custom ticker: AAPL
   Custom generator: svnn
   Custom z_dim: 5


## Test 3: Data Loading ✅

Test YFinanceSource for loading financial data.

In [9]:
# Use smaller date range for faster testing
test_data_cfg = DataConfig(
    ticker="SPY",
    start="2018-01-01",
    end="2018-12-31",
)

src = DefeatBetaSource(test_data_cfg)
df = src.fetch()
close = df["Close"].dropna()
logret = src.log_returns_from_close(df)

print("✅ Data loaded successfully")
print(f"   DataFrame shape: {df.shape}")
print(f"   Close prices: {len(close)} days")
print(f"   Log returns: {len(logret)} observations")
print(f"   Mean return: {logret.mean():.6f}")
print(f"   Std return: {logret.std():.6f}")
print(f"   Kurtosis: {kurtosis(logret, fisher=False, bias=False):.3f}")

[DATA] Downloading SPY from DefeatBeta API...


[nltk_data] Downloading package punkt_tab to /tmp/defeatbeta/nltk...
[nltk_data]   Package punkt_tab is already up-to-date!


______      __           _    ______      _        
|  _  \    / _|         | |   | ___ \    | |       
| | | |___| |_ ___  __ _| |_  | |_/ / ___| |_ __ _ 
| | | / _ \  _/ _ \/ _` | __| | ___ \/ _ \ __/ _` |
| |/ /  __/ ||  __/ (_| | |_  | |_/ /  __/ || (_| |
|___/ \___|_| \___|\__,_|\__| \____/ \___|\__\__,_|
📈:: Data Update Time ::	2026-02-06 ::
📈:: Software Version ::	0.0.41      ::
[DATA] Saved to cache: data/defeatbeta_SPY_2018-01-01_2018-12-31_1d.csv
✅ Data loaded successfully
   DataFrame shape: (251, 6)
   Close prices: 251 days
   Log returns: 250 observations
   Mean return: -0.000291
   Std return: 0.010812
   Kurtosis: 6.194


## Test 4: Preprocessing ✅

Test LambertW transformation.

In [11]:
# Fit preprocessor
pre = LambertWPreprocessor(pre_cfg).fit(logret)
summary = pre.summary()

print("✅ Preprocessor fitted")
print(f"   Use Lambert: {summary['use_lambert']}")
print(f"   Delta_hat: {summary['delta_hat']:.4f}")
print(f"   Lambda mu: {summary['lam_mu']:.4f}")
print(f"   Lambda sigma: {summary['lam_sigma']:.4f}")

# Transform
r_train = pre.transform(logret)
print(f"\n✅ Data transformed")
print(f"   Original: mean={logret.mean():.4f}, std={logret.std():.4f}")
print(f"   Transformed: mean={r_train.mean():.4f}, std={r_train.std():.4f}")

# Inverse transform
r_back = pre.inverse_transform(r_train)
diff = np.abs(logret - r_back).max()
print(f"\n✅ Inverse transform")
print(f"   Max difference: {diff:.2e} (should be ~0)")
assert diff < 1e-8, "Inverse transform not accurate!"

✅ Preprocessor fitted
   Use Lambert: True
   Delta_hat: 0.2578
   Lambda mu: 0.0831
   Lambda sigma: 0.6584

✅ Data transformed
   Original: mean=-0.0003, std=0.0108
   Transformed: mean=0.0000, std=1.0000

✅ Inverse transform
   Max difference: 5.31e-09 (should be ~0)


## Test 5: Dataset Building ✅

Test window creation and TensorFlow dataset.

In [12]:
test_ds_cfg = DatasetConfig(
    window_len=127,
    batch_size=16,  # Smaller for testing
    weighted_sampling=True,
    seed=42,
)

ds_builder = DatasetBuilder(test_ds_cfg)
train_ds, X_windows, steps_per_epoch = ds_builder.build(r_train)

print("✅ Dataset created")
print(f"   Window shape: {X_windows.shape}")
print(f"   Steps per epoch: {steps_per_epoch}")

# Test iteration
for batch in train_ds.take(1):
    print(f"\n✅ Dataset iteration works")
    print(f"   Batch shape: {batch.shape}")
    print(f"   Batch dtype: {batch.dtype}")
    print(f"   Batch range: [{batch.numpy().min():.3f}, {batch.numpy().max():.3f}]")
    break

✅ Dataset created
   Window shape: (124, 127, 1)
   Steps per epoch: 7

✅ Dataset iteration works
   Batch shape: (16, 127, 1)
   Batch dtype: <dtype: 'float32'>
   Batch range: [-2.528, 2.751]


## Test 6: Model Building ✅

Test both generator types and discriminator.

In [13]:
# Test Pure TCN generator
model_cfg_tcn = ModelConfig(
    generator_type="pure_tcn",
    z_dim=3,
    g_ch=32,  # Smaller for testing
)

netG_tcn = build_generator(model_cfg_tcn)
print("✅ Pure TCN Generator built")
print(f"   Model name: {netG_tcn.name}")
print(f"   Parameters: {netG_tcn.count_params():,}")

# Test forward pass
z_test = tf.random.normal([2, 150, 3])
out_tcn = netG_tcn(z_test, training=False)
print(f"   Input shape: {z_test.shape}")
print(f"   Output shape: {out_tcn.shape}")
print(f"   Output range: [{out_tcn.numpy().min():.3f}, {out_tcn.numpy().max():.3f}]")

# Test SVNN generator
model_cfg_svnn = ModelConfig(
    generator_type="svnn",
    z_dim=3,
    g_ch=32,
)

netG_svnn = build_generator(model_cfg_svnn)
print("\n✅ SVNN Generator built")
print(f"   Model name: {netG_svnn.name}")
print(f"   Parameters: {netG_svnn.count_params():,}")

out_svnn = netG_svnn(z_test, training=False)
print(f"   Output shape: {out_svnn.shape}")
print(f"   Output range: [{out_svnn.numpy().min():.3f}, {out_svnn.numpy().max():.3f}]")

# Test discriminator
model_cfg_d = ModelConfig(d_ch=32)
netD = build_discriminator(model_cfg_d)
print("\n✅ Discriminator built")
print(f"   Model name: {netD.name}")
print(f"   Parameters: {netD.count_params():,}")

# Test discriminator forward pass
x_test = tf.random.normal([2, 127, 1])
d_out = netD(x_test, training=False)
print(f"   Input shape: {x_test.shape}")
print(f"   Output shape: {d_out.shape}")
print(f"   Output range: [{d_out.numpy().min():.3f}, {d_out.numpy().max():.3f}]")

✅ Pure TCN Generator built
   Model name: G_PURE_TCN
   Parameters: 74,801
   Input shape: (2, 150, 3)
   Output shape: (2, 150, 1)
   Output range: [-3.244, 3.495]

✅ SVNN Generator built
   Model name: G_SVNN
   Parameters: 74,834
   Output shape: (2, 150, 1)
   Output range: [-3.996, 3.999]

✅ Discriminator built
   Model name: D
   Parameters: 73,073
   Input shape: (2, 127, 1)
   Output shape: (2, 1)
   Output range: [-2.421, -2.253]


## Test 7: Training Components ✅

Test learning rate schedule and trainer initialization.

In [14]:
# Test learning rate schedule
lr_schedule = EpochDecay(
    lr0=1e-4,
    steps_per_epoch_effective=100,
    decay_start=50,
    decay_rate=0.98,
    min_lr=1e-6,
)

print("✅ LR Schedule created")
print(f"   Step 0 (epoch 0): {lr_schedule(0):.2e}")
print(f"   Step 5000 (epoch 50): {lr_schedule(5000):.2e}")
print(f"   Step 10000 (epoch 100): {lr_schedule(10000):.2e}")

# Compute target stats for trainer
real_windows = X_windows.astype(np.float32)
N_TGT = min(100, len(real_windows))  # Small for testing
idx = np.random.RandomState(42).choice(len(real_windows), size=N_TGT, replace=False)
real_sub = tf.constant(real_windows[idx], dtype=tf.float32)

acf_abs_real_tgt = tf.reduce_mean(acf_tf(tf.abs(real_sub), 40), axis=0)
acf_sq_real_tgt = tf.reduce_mean(acf_tf(tf.square(real_sub), 40), axis=0)
lev_real_tgt = tf.reduce_mean(leverage_tf(real_sub, 40), axis=0)

print("\n✅ Target stats computed")
print(f"   ACF abs shape: {acf_abs_real_tgt.shape}")
print(f"   ACF sq shape: {acf_sq_real_tgt.shape}")
print(f"   Leverage shape: {lev_real_tgt.shape}")

# Test trainer initialization
test_model_cfg = ModelConfig(generator_type="pure_tcn", g_ch=32, d_ch=32)
test_train_cfg = TrainConfig(epochs=5)

trainer = WGANGPTrainer(
    model_cfg=test_model_cfg,
    train_cfg=test_train_cfg,
    debug_cfg=DebugConfig(verbose=False),  # Quiet for testing
    window_len=test_ds_cfg.window_len,
    steps_per_epoch=steps_per_epoch,
    vol_lags=40,
    lev_lags=40,
    acf_abs_real_tgt=acf_abs_real_tgt,
    acf_sq_real_tgt=acf_sq_real_tgt,
    lev_real_tgt=lev_real_tgt,
)

print("\n✅ Trainer initialized")
print(f"   Generator params: {trainer.netG.count_params():,}")
print(f"   Discriminator params: {trainer.netD.count_params():,}")
print(f"   Window length: {trainer.window_len}")
print(f"   Burn-in: {trainer.burn_in}")

✅ LR Schedule created
   Step 0 (epoch 0): 1.00e-04
   Step 5000 (epoch 50): 1.00e-04
   Step 10000 (epoch 100): 3.64e-05

✅ Target stats computed
   ACF abs shape: (40,)
   ACF sq shape: (40,)
   Leverage shape: (40,)

✅ Trainer initialized
   Generator params: 74,801
   Discriminator params: 73,073
   Window length: 127
   Burn-in: 126


## Test 8: Evaluation & Metrics ✅

Test metric computations and evaluator.

In [15]:
# Test TensorFlow metrics
test_data = tf.random.normal([10, 100, 1])

acf_result = acf_tf(test_data, lags=20)
print("✅ ACF (TensorFlow)")
print(f"   Input shape: {test_data.shape}")
print(f"   Output shape: {acf_result.shape}")
print(f"   Output range: [{acf_result.numpy().min():.3f}, {acf_result.numpy().max():.3f}]")

lev_result = leverage_tf(test_data, nlags=20)
print("\n✅ Leverage (TensorFlow)")
print(f"   Output shape: {lev_result.shape}")
print(f"   Output range: [{lev_result.numpy().min():.3f}, {lev_result.numpy().max():.3f}]")

# Test NumPy metrics
test_series = np.random.randn(200)

acf_np = acf_vec(test_series, S=20)
print("\n✅ ACF (NumPy)")
print(f"   Input length: {len(test_series)}")
print(f"   Output shape: {acf_np.shape}")
print(f"   Output range: [{acf_np.min():.3f}, {acf_np.max():.3f}]")

lev_np = lev_vec(test_series, S=20)
print("\n✅ Leverage (NumPy)")
print(f"   Output shape: {lev_np.shape}")
print(f"   Output range: [{lev_np.min():.3f}, {lev_np.max():.3f}]")

# Test PaperEvaluator
evaluator = PaperEvaluator(
    real_series=logret,
    preproc=pre,
    train_cfg=test_train_cfg,
    dy_base_t=len(logret),
)

print("\n✅ PaperEvaluator created")
print(f"   Real mean: {evaluator.real_mean:.6f}")
print(f"   Real std: {evaluator.real_std:.6f}")
print(f"   Real quantiles: {evaluator.real_qs}")

✅ ACF (TensorFlow)
   Input shape: (10, 100, 1)
   Output shape: (10, 20)
   Output range: [-0.305, 0.268]

✅ Leverage (TensorFlow)
   Output shape: (10, 20)
   Output range: [-0.289, 0.284]

✅ ACF (NumPy)
   Input length: 200
   Output shape: (20,)
   Output range: [-0.134, 0.169]

✅ Leverage (NumPy)
   Output shape: (20,)
   Output range: [-0.140, 0.109]

✅ PaperEvaluator created
   Real mean: -0.000291
   Real std: 0.010812
   Real quantiles: [-0.0325648  -0.02169678  0.01465034  0.0222086 ]


## Test 9: Utilities ✅

Test random seeds, I/O, and inference utilities.

In [16]:
# Test seed setting
set_all_seeds(42)
r1 = np.random.rand(5)
set_all_seeds(42)
r2 = np.random.rand(5)

print("✅ Seed management")
print(f"   First run: {r1}")
print(f"   Second run: {r2}")
print(f"   Identical: {np.allclose(r1, r2)}")
assert np.allclose(r1, r2), "Seeds not working!"

# Test weights metadata
import tempfile
with tempfile.TemporaryDirectory() as tmpdir:
    test_weights_path = os.path.join(tmpdir, "test.weights.h5")
    
    # Write metadata
    meta_path = write_weights_meta(test_weights_path, test_model_cfg)
    
    print("\n✅ Weights metadata")
    print(f"   Weights path: {test_weights_path}")
    print(f"   Metadata path: {meta_path}")
    print(f"   Metadata exists: {os.path.exists(meta_path)}")
    
    # Read it back
    import json
    with open(meta_path, 'r') as f:
        meta = json.load(f)
    print(f"   Generator type in meta: {meta['generator_type']}")
    print(f"   Z-dim in meta: {meta['z_dim']}")

# Test generation (without loading weights - just test the function)
print("\n✅ Generation utilities exist and are callable")
print(f"   generate_M_paths_raw: {callable(generate_M_paths_raw)}")
print(f"   build_and_load_generator: {callable(build_and_load_generator)}")

✅ Seed management
   First run: [0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]
   Second run: [0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]
   Identical: True

✅ Weights metadata
   Weights path: /tmp/tmpcp4b705d/test.weights.h5
   Metadata path: /tmp/tmpcp4b705d/test.meta.json
   Metadata exists: True
   Generator type in meta: pure_tcn
   Z-dim in meta: 3

✅ Generation utilities exist and are callable
   generate_M_paths_raw: True
   build_and_load_generator: True


## Test 10: Mini Training Run ✅

Run a very short training (5 epochs) to ensure everything works end-to-end.

In [17]:
print("Starting mini training run (5 epochs)...\n")

# Use very small config for fast testing
mini_train_cfg = TrainConfig(
    epochs=5,
    n_critic=3,
    pretrain_d_epochs=1,
    sel_every=2,
    sel_m=10,  # Very small
    sel_ttilde=200,  # Very small
)

mini_debug_cfg = DebugConfig(
    verbose=True,
    debug_tails=False,
    debug_raw_every=10,
)

# Create new trainer with mini config
mini_trainer = WGANGPTrainer(
    model_cfg=test_model_cfg,
    train_cfg=mini_train_cfg,
    debug_cfg=mini_debug_cfg,
    window_len=test_ds_cfg.window_len,
    steps_per_epoch=min(10, steps_per_epoch),  # Only 10 steps per epoch
    vol_lags=40,
    lev_lags=40,
    acf_abs_real_tgt=acf_abs_real_tgt,
    acf_sq_real_tgt=acf_sq_real_tgt,
    lev_real_tgt=lev_real_tgt,
)

# Create temp output directory
import tempfile
with tempfile.TemporaryDirectory() as tmpdir:
    result = mini_trainer.fit(
        train_ds=train_ds,
        evaluator=evaluator,
        out_dir=tmpdir,
        real_logret=logret,
        close_series=close,
        n_plot_runs=3,  # Very small
        n_plot_paths=3,  # Very small
        show_plots=False,
        save_plots=False,
        run_seed=42,
    )
    
    print("\n" + "="*60)
    print("✅ MINI TRAINING COMPLETE!")
    print("="*60)
    print(f"Generator type: {result['gen_type']}")
    print(f"Best epoch: {result['best_epoch']}")
    print(f"Best score: {result['best_score']:.6f}")
    print(f"Weights saved: {os.path.exists(result['best_weights_path'])}")
    print(f"Seed: {result['seed']}")

Starting mini training run (5 epochs)...

2026-02-14 00:19:01 INFO quantgan.training.trainer MainThread - [MODEL] gen_type=pure_tcn CAUSAL=True SKIPS=True T_D=127 Z_DIM=3 KERNEL=2 DILATIONS=[1, 1, 2, 4, 8, 16, 32] G_vars=86 D_vars=58
2026-02-14 00:19:11 INFO quantgan.training.trainer MainThread - [PRETRAIN D] E00 lossD=19.36 W=-2.849 GP=2.221 gp_norm_mean=2.443 gp_norm_max=3.674


2026-02-14 00:19:12.534823: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


2026-02-14 00:19:12 INFO quantgan.training.trainer MainThread - [E00] -> saved BEST generator (paper_score=1.06675)
2026-02-14 00:19:12 INFO quantgan.training.trainer MainThread - [E00] done lastD=0 lastG=0 PAPER_score=1.06675 acf_abs=0.198 acf_x=0.163 acf_sq=0.196 lev=3.01 dy_sum=835 DY1=824 DY5=1.57 DY20=5.19 DY100=4.01 | raw_mean=-0.0871 raw_std=0.0417 mean_err=8.03 std_err=2.86 q_err=8.49
2026-02-14 00:19:20 INFO quantgan.training.trainer MainThread - [E01 B0000] lossD=3.396 (W=-5.905, GP=0.9301) | lossG=7.044 | gp_norm_mean=1.928 gp_norm_max=2.649 | mom=11.26 drift=9.351 acf|r|=0.009908 acf r2=0.01466 lev=0.007403
2026-02-14 00:19:20 INFO quantgan.training.trainer MainThread -   D(real) mean: -0.8315 D(fake) mean: -6.737
2026-02-14 00:19:21 INFO quantgan.training.trainer MainThread - [E01] done lastD=11.32 lastG=3.209 (no paper-eval this epoch) | raw_mean=-0.015 raw_std=0.0194 mean_err=1.36 std_err=0.796 q_err=2.42


2026-02-14 00:19:21.597982: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


2026-02-14 00:19:22 INFO quantgan.training.trainer MainThread - [E02 B0000] lossD=7.729 (W=-1.9, GP=0.9629) | lossG=2.902 | gp_norm_mean=1.905 gp_norm_max=2.825 | mom=1.452 drift=1.258 acf|r|=0.01172 acf r2=0.01557 lev=0.008348
2026-02-14 00:19:22 INFO quantgan.training.trainer MainThread -   D(real) mean: -1.216 D(fake) mean: -3.116
2026-02-14 00:19:23 INFO quantgan.training.trainer MainThread - [E02] -> saved BEST generator (paper_score=0.888379)
2026-02-14 00:19:23 INFO quantgan.training.trainer MainThread - [E02] done lastD=5.553 lastG=0.8895 PAPER_score=0.888379 acf_abs=0.198 acf_x=0.166 acf_sq=0.193 lev=2.89 dy_sum=173 DY1=22.6 DY5=21.1 DY20=34.8 DY100=94.5 | raw_mean=0.000937 raw_std=0.0125 mean_err=0.114 std_err=0.153 q_err=0.535
2026-02-14 00:19:24 INFO quantgan.training.trainer MainThread - [E03 B0000] lossD=8.171 (W=0.08104, GP=0.809) | lossG=0.9525 | gp_norm_mean=1.850 gp_norm_max=2.408 | mom=0.1489 drift=0.0002653 acf|r|=0.01012 acf r2=0.01519 lev=0.006514
2026-02-14 00:19

2026-02-14 00:19:25.548574: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


2026-02-14 00:19:25 INFO quantgan.training.trainer MainThread - [E04 B0000] lossD=3.756 (W=-0.1399, GP=0.3896) | lossG=1.6 | gp_norm_mean=1.577 gp_norm_max=1.983 | mom=0.1946 drift=0.08727 acf|r|=0.01003 acf r2=0.01468 lev=0.006475
2026-02-14 00:19:25 INFO quantgan.training.trainer MainThread -   D(real) mean: -1.099 D(fake) mean: -1.239
2026-02-14 00:19:27 INFO quantgan.training.trainer MainThread - [E04] done lastD=1.03 lastG=1.502 PAPER_score=0.915477 acf_abs=0.191 acf_x=0.16 acf_sq=0.187 lev=2.87 dy_sum=358 DY1=23 DY5=25.5 DY20=40.2 DY100=269 | raw_mean=-0.00233 raw_std=0.0117 mean_err=0.189 std_err=0.0858 q_err=0.265
2026-02-14 00:19:27 INFO quantgan.training.trainer MainThread - 
=== BEST PAPER MODEL ===
2026-02-14 00:19:27 INFO quantgan.training.trainer MainThread - best_epoch=2 best_paper_score=0.888379
2026-02-14 00:19:27 INFO quantgan.training.trainer MainThread - best_parts: {'acf_x': 0.16569087203827823, 'acf_abs': 0.19769072541550478, 'acf_sq': 0.19320720587489532, 'lev': 

## Test Summary 📊

If you've reached this point without errors, all components are working correctly!

In [18]:
print("="*60)
print("🎉 ALL TESTS PASSED! 🎉")
print("="*60)
print("\nComponents tested:")
print("  ✅ Package imports")
print("  ✅ Configuration classes")
print("  ✅ Data loading (YFinance)")
print("  ✅ Preprocessing (LambertW)")
print("  ✅ Dataset building")
print("  ✅ Model building (Pure TCN + SVNN + Discriminator)")
print("  ✅ Training components (Schedule + Trainer)")
print("  ✅ Evaluation & Metrics")
print("  ✅ Utilities (Seeds, I/O, Inference)")
print("  ✅ End-to-end mini training")
print("\nYour QuantGAN installation is ready to use!")
print("\nNext steps:")
print("  📓 Open notebooks/01_training.ipynb for full training")
print("  📓 Open notebooks/02_evaluation.ipynb for evaluation")
print("  📚 Read QUICKSTART.md for more examples")

🎉 ALL TESTS PASSED! 🎉

Components tested:
  ✅ Package imports
  ✅ Configuration classes
  ✅ Data loading (YFinance)
  ✅ Preprocessing (LambertW)
  ✅ Dataset building
  ✅ Model building (Pure TCN + SVNN + Discriminator)
  ✅ Training components (Schedule + Trainer)
  ✅ Evaluation & Metrics
  ✅ Utilities (Seeds, I/O, Inference)
  ✅ End-to-end mini training

Your QuantGAN installation is ready to use!

Next steps:
  📓 Open notebooks/01_training.ipynb for full training
  📓 Open notebooks/02_evaluation.ipynb for evaluation
  📚 Read QUICKSTART.md for more examples
